# 02 — Tracing a sensorimotor pathway

Phase 1 turned the brain into a signed sparse matrix. This notebook asks what that matrix *does*:
how does a signal from the eyes, antennae or tongue reach the **descending neurons (DNs)**, the
~1,300 neurons that carry commands from the brain down the neck to the body? DNs are the
controller's output layer, and the interface Phase 4 will connect to a simulated fly in MuJoCo.

Two parts:
1. **Flagship circuit:** looming visual input to the giant fiber (DNp01), the fly's fastest escape.
2. **Brain-wide:** how far every sensory modality is from the DNs.

All algorithms live in `src/c2c/analysis/pathways.py` (tested in `tests/test_pathways.py`).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from c2c.analysis import graph_stats as gs
from c2c.analysis import pathway_plots as pp
from c2c.analysis import pathways as pw
from c2c.data import annotations as an
from c2c.data.graph import load_graph
from c2c.paths import FIGURES, PROCESSED

gs.apply_style()
FIGURES.mkdir(parents=True, exist_ok=True)

## Load the graph and join the annotations

In [ ]:
g = load_graph(PROCESSED / "graph_v783_min5.npz")
ann = an.align_to_graph(an.load_annotations(), g)   # row k describes matrix index k
an.coverage(ann)

In [ ]:
# What kinds of neurons are in the >=5-synapse graph?
ann["super_class"].value_counts(dropna=False).to_frame("neurons")

## Part 1 — Looming to escape: the giant fiber

When something approaches fast (a looming shadow, a swatter), a fly can be airborne in a few
milliseconds. The fastest route runs through the **giant fiber** (DNp01), one per side, which
drives the jump muscle in the body. Two visual projection neuron types, **LPLC2** and **LC4**,
are known to encode looming; we check what the connectome says.

In [ ]:
gf = an.select(ann, cell_type="DNp01")
ann.loc[gf, ["root_id", "cell_type", "hemibrain_type", "side", "top_nt"]]

Note the predicted transmitter: the right giant fiber is predicted **glutamatergic**, the left
**cholinergic**. The giant fiber is a well-studied cholinergic neuron that also uses electrical
synapses, so the right-side prediction is almost certainly wrong. That changes the sign of its
*outgoing* edges in `W` (which matter little here, since its targets are in the body, outside this
brain dataset), and it is a useful reminder that signs are predictions with error bars.

In [ ]:
F = pw.input_fractions(g)        # w[i, j] = share of i's input that comes from j
edges = pw.edge_table(g)
edges["fraction"] = F.data       # same storage order as counts, so this lines up

def input_breakdown(target, labels=ann["cell_type"], top=8):
    rows = edges[edges["post"] == target].copy()
    rows["type"] = labels.to_numpy()[rows["pre"]]
    return (rows.groupby("type")
                .agg(neurons=("pre", "size"), synapses=("synapses", "sum"),
                     fraction=("fraction", "sum"), sign=("sign", "mean"))
                .sort_values("fraction", ascending=False).head(top).round(3))

for k in gf:
    print(f"giant fiber, {ann.at[k, 'side']} side")
    display(input_breakdown(k))

**Check passed:** LC4 and LPLC2 are the two largest input types of both giant fibers, together
roughly a fifth of their input, matching the electrophysiology literature (von Reyn et al. 2017,
Ache et al. 2019). The pipeline recovers known biology without being told what to look for.

### From photoreceptors to the giant fiber
Hop distance first (breadth-first search), then the *strongest* path, i.e. the chain that
maximises the product of input fractions (Dijkstra on −log fraction).

In [ ]:
photo = an.select(ann, cell_type=["R1-6", "R7", "R8"])   # compound-eye photoreceptors
d_photo = pw.hop_distance(g, photo)
print("hops from photoreceptors to each giant fiber:", d_photo[gf])

sp = pw.strongest_paths(g, photo, fractions=F)
ct = ann["cell_type"].fillna("?").to_numpy()
for k in gf:
    p = pw.path_effect(g, sp.path_to(k), F)
    print(f"{ann.at[k, 'side']:>5}: " + " -> ".join(ct[p.nodes]),
          f"| fractions {np.round(p.fraction, 3)} | strength {p.strength:.1e} | sign {p.net_sign:+d}")

At single-neuron resolution the strongest chain skips the famous motion detectors T4/T5: each T4
neuron gives an LPLC2 only a few synapses, so no *single* T4 edge looks strong. Populations matter,
so we rebuild the graph with one node per **cell type** and run the same algorithms on it.

In [ ]:
gt, type_names = pw.group_graph(g, ann["cell_type"])
tix = {name: k for k, name in enumerate(type_names)}
Ft = pw.input_fractions(gt)
print(f"cell-type graph: {gt.n:,} types, {gt.counts.nnz:,} type-to-type edges")

def type_path(sources, target):
    s = pw.strongest_paths(gt, [tix[x] for x in sources], fractions=Ft)
    p = pw.path_effect(gt, s.path_to(tix[target]), Ft)
    return " -> ".join(type_names[p.nodes]), p

for src in (["R1-6"], ["R7"], ["R8"]):
    route, p = type_path(src, "DNp01")
    print(f"{src[0]:>4}: {route}   strength {p.strength:.1e}, net sign {p.net_sign:+d}")

### Escape is a population, not one neuron
Which DNs get the largest share of their input from LPLC2 + LC4? Row 1 of `multihop_effect` is
exactly that (the direct, one-hop input fraction); row 2 adds every two-hop route, signed.

In [ ]:
loom = an.select(ann, cell_type=["LPLC2", "LC4"])
x = pw.multihop_effect(g, loom, n_hops=2, fractions=F)
dn = an.select(ann, super_class="descending")
loom_dn = (pd.DataFrame({"cell_type": ct[dn], "side": ann["side"].to_numpy()[dn],
                         "direct": x[1, dn], "two_hop": x[2, dn]})
             .assign(total=lambda t: t.direct + t.two_hop)
             .sort_values("total", ascending=False))
loom_dn.head(10).round(3)

The giant fiber is not even the top target: **DNp04 gets about 60% of its input from LPLC2/LC4**,
and DNp02 and DNp11 are in the same league. These are the other looming-responsive DNs that drive
the slower, more controlled takeoffs. For a robot, that is the interesting part: a *graded*
escape choice spread over several output channels, not a single panic button.

### The circuit diagram
Cell types on the strongest routes from each photoreceptor type to the giant fiber and DNp04,
plus the main inputs of the looming neurons and of the escape DNs. Edges shown if they supply at
least 3% of the target type's input; only feed-forward edges are drawn (feedback exists, but it
hides the main flow). Line width = share of the target's input.

In [ ]:
escape_dns = ["DNp01", "DNp04", "DNp02", "DNp11"]
nodes = set(escape_dns) | {"LPLC2", "LC4"}
for src in ("R1-6", "R7", "R8"):
    for dn_type in ("DNp01", "DNp04"):
        nodes |= set(type_path([src], dn_type)[0].split(" -> "))

def top_inputs(t, k):
    row = Ft.getrow(tix[t])
    order = np.argsort(-row.data)[:k]
    return set(type_names[row.indices[order]])

for t in ("LPLC2", "LC4"):
    nodes |= top_inputs(t, 3)
for t in escape_dns:
    nodes |= top_inputs(t, 2)

idx = np.array(sorted(tix[n] for n in nodes))
sub = pw.subgraph(gt, idx)
sub_names = type_names[idx]
e = pw.edge_table(sub)
e["fraction"] = pw.input_fractions(gt).tocsr()[idx][:, idx].tocsr()[e.post, e.pre].A1
e = e[e["fraction"] >= 0.03].assign(pre=lambda t: sub_names[t.pre], post=lambda t: sub_names[t.post])
e = e[e.pre != e.post]

# Columns follow the anatomy: photoreceptors | optic lobe (split by depth) | visual projection
# neurons | central brain | descending neurons. Depth inside the optic lobe = type-level hops.
hops = pw.hop_distance(gt, [tix[s] for s in ("R1-6", "R7", "R8")])
type_class = ann.groupby("cell_type")["super_class"].agg(lambda s: s.mode().iat[0])
column_of = {"sensory": 0, "visual_projection": 3, "central": 4, "descending": 5}
layer = {}
for n in nodes:
    sc_ = type_class[n]
    layer[n] = column_of[sc_] if sc_ in column_of else min(int(hops[tix[n]]), 2)
e = e[[layer[a] < layer[b] for a, b in zip(e.pre, e.post)]]   # feed-forward edges only
names = ["photoreceptors", "optic lobe I", "optic lobe II", "visual projection",
         "central brain", "descending"]
print(f"{len(nodes)} cell types, {len(e)} edges; layers: {sorted(set(layer.values()))}")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6.5))
pp.plot_layered_circuit(layer, e, ax=ax, highlight={"LPLC2", "LC4", "DNp01", "DNp04"},
                        layer_names=[names[x] for x in sorted(set(layer.values()))])
ax.set_title("From photons to takeoff: the looming-escape circuit (cell-type level)", pad=28)
fig.savefig(FIGURES / "looming_escape_circuit.png")
plt.show()

## Part 2 — Every sense, every descending neuron

Now the brain-wide version: start from **all** sensory neurons and measure how far each neuron is.

In [ ]:
sensory = an.select(ann, super_class=list(an.SENSORY_SUPER_CLASSES))
d_all = pw.hop_distance(g, sensory)
print(f"{len(sensory):,} sensory seeds; unreachable neurons: {(d_all == pw.UNREACHED).sum():,}")
pd.Series(d_all[dn]).value_counts().sort_index().rename("DNs").to_frame()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
pp.plot_hop_histogram(d_all[d_all > 0], d_all[dn][d_all[dn] > 0], "descending neurons", ax=ax)
ax.set_title("How far is the brain from its senses?")
fig.savefig(FIGURES / "hops_from_sensory.png")
plt.show()

Almost every neuron is within 3 hops of *some* sensory neuron. Hubs make the brain a small world,
which is also why raw hop counts are a blunt instrument; the information-flow layers below fix that.

### Which senses reach the descending neurons fastest?
One BFS, one Dijkstra and one probabilistic traversal per modality.

In [ ]:
modality = an.sensory_modality(ann)
dn_mask = np.zeros(g.n, bool); dn_mask[dn] = True
rows, hop_share, strength = [], {}, {}
for m in modality.dropna().unique():
    seeds = np.flatnonzero(modality.eq(m).fillna(False).to_numpy())
    d = pw.hop_distance(g, seeds)[dn]
    reached = d[d > 0]
    hop_share[m] = pd.Series(reached).value_counts(normalize=True)
    strength[m] = pw.strongest_paths(g, seeds, fractions=F).strength[dn]
    layer_m, _ = pw.information_flow(g, seeds, n_runs=64, fractions=F, seed=0)
    rows.append({"modality": m, "sensory neurons": len(seeds),
                 "DNs at 1 hop": (d == 1).mean(), "DNs within 2 hops": ((d >= 1) & (d <= 2)).mean(),
                 "mean hops": reached.mean(), "median info-flow layer": np.nanmedian(layer_m[dn])})
speed = pd.DataFrame(rows).set_index("modality").sort_values("mean hops")
speed.round(3)

In [ ]:
heat = pd.DataFrame(hop_share).T.reindex(speed.index).fillna(0)
heat.columns = heat.columns.astype(int)
heat = heat.reindex(columns=range(1, heat.columns.max() + 1), fill_value=0)
heat = heat.loc[:, heat.max() >= 0.005]   # drop hop counts no modality uses
fig, ax = plt.subplots(figsize=(7, 4.5))
pp.plot_modality_heatmap(heat, ax=ax)
ax.set_title("Hops from each sense to the descending neurons")
fig.savefig(FIGURES / "modality_hops_to_dns.png")
plt.show()

Touch and body sense (mechanosensory, Johnston's organ in the antenna, and the ascending sensory
neurons from the legs) are wired closest to the motor output: most DNs are two hops away, and over
a hundred get direct input. Vision, the largest sense by far, sits three to four hops out, behind
the optic lobes, which spend those layers turning pixels into features like "something is looming".

### Which sense dominates each DN?
For each DN, the modality whose strongest path into it is strongest.

In [ ]:
S = pd.DataFrame(strength)
dominant = S.idxmax(axis=1)
dn_family = pd.Series(ct[dn]).str.extract(r"^(DN[a-z])")[0].fillna("other")
pd.crosstab(dn_family.to_numpy(), dominant.to_numpy(), margins=True)

### Probabilistic information-flow layers
The traversal model of Schlegel et al. (2021): a neuron passes the signal on with probability
min(1, input fraction / 0.3), and its layer is the average step at which it gets activated over many
random runs. A single weak synapse no longer makes a neuron look close.

In [ ]:
layer_all, reach_all = pw.information_flow(g, sensory, n_runs=128, fractions=F, seed=0)
groups = ann["super_class"]
order = (pd.Series(layer_all).groupby(groups.to_numpy()).median()
           .drop(index=["sensory", "sensory_ascending"], errors="ignore").sort_values().index.tolist())
fig, ax = plt.subplots(figsize=(6.5, 4))
pp.plot_layer_by_group(pd.Series(layer_all), groups, order, ax=ax,
                       xlabel="information-flow layer from all sensory neurons")
ax.set_title("Where each class sits in the sensory-to-motor flow")
fig.savefig(FIGURES / "information_flow_layers.png")
plt.show()
print("median layer of DNs:", round(float(np.nanmedian(layer_all[dn])), 2))

## Takeaways

*(Fill in with your own numbers after running; the shape of the result should match.)*

- **The pipeline recovers known biology.** LPLC2 and LC4 are the top two inputs of the giant fiber.
- **Escape is distributed.** DNp04 receives about three times the giant fiber's share of looming input.
- **Touch is closest to action, vision farthest.** Mechanosensory DNs are mostly 2 hops away; visual
  ones 3–4, after the optic lobes have extracted features.
- **Hop counts saturate, flow layers don't.** Almost everything is ≤3 hops from a sense, so the
  probabilistic layering is the better ruler for "how deep" a neuron sits.